# Lab 09: Azure AI Foundry — Everything Out of the Box

## 🎯 The "Aha" Moment

In Labs 01–07, you built every component of an AI Agent Platform **by hand**:

| Lab | What You Built | Lines of Code |
|-----|---------------|---------------|
| Lab 01 | ReAct loop, tool calls | ~80 lines |
| Lab 03 | Memory, RAG pipeline, vector store | ~120 lines |
| Lab 04 | Orchestration (sequential, parallel) | ~100 lines |
| Lab 05 | Tool registry, safety guardrails | ~90 lines |
| Lab 06 | Evaluation pipeline | ~80 lines |

**Now watch Azure AI Foundry do ALL of that for you.**

This notebook has three parts:

| Part | What | Replaces |
|------|------|----------|
| **Part 1** | Build a Foundry Agent with File Search | Labs 01 + 03 |
| **Part 2** | Run built-in evaluations (quality + safety + agent) | Lab 06 |
| **Part 3** | Instrument with OpenTelemetry tracing | Lab 08 |

> **Credits:** This lab is adapted from Microsoft's [Ignite 2025 PREL13 Workshop](https://github.com/microsoft/ignite25-PREL13-observe-manage-and-scale-agentic-ai-apps-with-microsoft-foundry) by **Nitya Narasimhan**, **Bethany Jepchumba**, and the Microsoft AI team.

---

## 🔧 Setup: Install Packages & Load Environment

Lab 00 already deployed your Azure AI Foundry resource. We just need the Foundry SDK packages.

In [1]:
# ──────────────────────────────────────────────────────
# Install Foundry-specific packages
#
# • azure-ai-projects    — Foundry SDK (agents, threads, runs)
# • azure-ai-evaluation  — Built-in evaluators (quality + safety)
# • azure-identity       — Auth with Azure (DefaultAzureCredential)
# • openai + openai-agents — OpenAI Agents framework (for Part 3)
# • opentelemetry-sdk    — Tracing core
# • opentelemetry-instrumentation-openai-agents-v2 — GenAI tracing
# • azure-monitor-opentelemetry-exporter — Export to App Insights
#
# Note: You may see dependency warnings from other packages
# in your environment (chainlit, litellm, semantic-kernel, etc.)
# These are safe to ignore — they don't affect this lab.
# ──────────────────────────────────────────────────────
import subprocess, sys

packages = [
    "azure-ai-projects", "azure-ai-evaluation", "azure-identity",
    "openai", "openai-agents", "python-dotenv", "rich",
    "opentelemetry-sdk", "opentelemetry-instrumentation-openai-agents-v2",
    "azure-monitor-opentelemetry-exporter",
]
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "--quiet"] + packages,
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
print("✅ All packages installed successfully.")

✅ All packages installed successfully.


In [1]:
# ──────────────────────────────────────────────────────
# Load environment from labs/.env (created by Lab 00)
# ──────────────────────────────────────────────────────
import os
from pathlib import Path
from dotenv import load_dotenv

env_path = Path("../.env")
if not env_path.exists():
    raise FileNotFoundError("No .env file found. Run Lab 00 first!")

load_dotenv(dotenv_path=env_path, override=True)

# Verify required variables
required = [
    "AZURE_OPENAI_ENDPOINT",
    "AZURE_OPENAI_API_KEY",
    "AZURE_OPENAI_DEPLOYMENT_GPT41",
    "AZURE_AI_FOUNDRY_PROJECT",
    "AZURE_AI_FOUNDRY_RESOURCE",
]

missing = [v for v in required if not os.getenv(v)]
if missing:
    print(f"❌ Missing env vars: {missing}")
    print("   Run Lab 00 setup first!")
else:
    print("✅ All environment variables loaded")
    print(f"   Endpoint:  {os.getenv('AZURE_OPENAI_ENDPOINT')[:40]}...")
    print(f"   Model:     {os.getenv('AZURE_OPENAI_DEPLOYMENT_GPT41')}")
    print(f"   Project:   {os.getenv('AZURE_AI_FOUNDRY_PROJECT')}")
    print(f"   Resource:  {os.getenv('AZURE_AI_FOUNDRY_RESOURCE')}")

✅ All environment variables loaded
   Endpoint:  https://ai-agentlabs-fk25xdiuwudo4.cogni...
   Model:     gpt-41
   Project:   agentlabs-project
   Resource:  ai-agentlabs-fk25xdiuwudo4


---

# Part 1: Build a Foundry Agent with File Search

## 🔄 Remember Lab 01?

In Lab 01, you built the ReAct loop from scratch — parsing tool calls, executing functions, feeding results back to the LLM, managing conversation history... **~80 lines of code.**

In Lab 03, you built a RAG pipeline — loading documents, chunking, embedding, creating a vector index, writing retrieval logic... **~120 lines.**

Watch how Azure AI Foundry Agent Service gives you **all of that** in a few lines:

```
What you built in Labs 01+03:          What Foundry handles:
─────────────────────────────          ──────────────────────
ReAct loop (while + tool calls)   →    Agents Service (managed orchestration)
Conversation history management   →    Threads (managed state)
Document loading + chunking       →    File Upload
Embedding + vector index          →    Vector Store (managed)
Retrieval logic                   →    File Search Tool (built-in)
Content safety checks             →    Content Filters (built-in)
```

## Step 1: Connect to Azure AI Foundry

The `AIProjectClient` is your gateway to everything in Foundry — agents, threads, evaluations, and more.

Compare this to Lab 01 where you needed separate clients for OpenAI, Cosmos DB, AI Search, etc.

In [ ]:
# ──────────────────────────────────────────────────────
# Connect to Azure AI Foundry
#
# The Microsoft PREL13 workshop uses the same deployment
# pattern as us: CognitiveServices/accounts/projects.
# The project endpoint comes from the Bicep output:
#   aiProject.properties.endpoints['AI Foundry API']
#
# We read it from AZURE_AI_FOUNDRY_ENDPOINT in .env.
# If not set (older deployment), we construct it manually.
# ──────────────────────────────────────────────────────

from azure.ai.agents import AgentsClient
from azure.identity import DefaultAzureCredential

# Use the official endpoint from Bicep output, or construct it
foundry_endpoint = os.getenv("AZURE_AI_FOUNDRY_ENDPOINT")

if not foundry_endpoint:
    # Fallback for older deployments that don't have the endpoint in .env
    foundry_resource = os.getenv("AZURE_AI_FOUNDRY_RESOURCE")
    foundry_project = os.getenv("AZURE_AI_FOUNDRY_PROJECT")
    foundry_endpoint = f"https://{foundry_resource}.services.ai.azure.com/api/projects/{foundry_project}"
    print(f"⚠️  AZURE_AI_FOUNDRY_ENDPOINT not in .env — using constructed URL")
    print(f"   Re-run infra/deploy.sh to get the official endpoint")

print(f"🔗 Connecting to: {foundry_endpoint}")

agents_client = AgentsClient(
    endpoint=foundry_endpoint,
    credential=DefaultAzureCredential(),
)

print(f"\n✅ Connected to Azure AI Foundry")
print(f"   Endpoint: {foundry_endpoint}")
print(f"\n📍 View agents at: https://ai.azure.com → Agents")

🔗 Connecting to: https://ai-agentlabs-fk25xdiuwudo4.services.ai.azure.com/api/projects/agentlabs-project

✅ Connected to Azure AI Foundry
   Resource: ai-agentlabs-fk25xdiuwudo4
   Project:  agentlabs-project

📍 View agents at: https://ai.azure.com → agentlabs-project → Agents


/Users/robenhai/.pyenv/versions/3.11.0/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.3.0) or chardet (7.1.0)/charset_normalizer (3.4.1) doesn't match a supported version!
  warnings.warn(


## Step 2: Create the Agent

In Lab 01, you wrote ~20 lines of system prompt handling + client configuration.

With Foundry Agent Service, you define the agent's personality and model in one call.
The service handles the orchestration loop, tool execution, and content safety **for you**.

In [3]:
# Create the agent — compare this to the ~80 lines in Lab 01!
model_deployment = os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT41", "gpt-41")

agent = agents_client.create_agent(
    model=model_deployment,
    name="Lab09-Agent",
    instructions="""You are a friendly and helpful customer service agent for Zava Hardware Store.

Your role is to assist customers with home improvement and hardware needs by:
- Answering questions about Zava's products (paint, tools, hardware)
- Helping customers find the right items for their projects
- Providing accurate product information (names, SKUs, prices, descriptions, stock levels)
- Being polite, factual, and helpful

Communication Style:
1. Always greet customers warmly
2. Use relevant emojis (🎨 for paint, 🔨 for tools, 🔧 for hardware)
3. Provide accurate, factual information based on product documents
4. End with helpful suggestions
5. If asked about unrelated topics, politely redirect to Zava products

Remember: You represent Zava Hardware Store — be professional, friendly, and focused on home improvement needs."""
)

print(f"✅ Agent created: {agent.name} (ID: {agent.id})")

✅ Agent created: Lab09-Agent (ID: asst_FYInV4gmbxrZGRB5JmereStE)


## Step 3: Upload Product Files & Create Vector Store

Remember Lab 03 where you:
1. Loaded documents
2. Split them into chunks
3. Generated embeddings with Azure OpenAI
4. Created an AI Search index
5. Uploaded vectors
6. Built retrieval logic

**With Foundry, you just upload files and create a vector store. That's it.**

In [ ]:
from azure.ai.agents.models import FilePurpose

# Upload product files — Foundry handles chunking & embedding automatically
print("📤 Uploading product files...")
product_dir = Path("data/products")
uploaded_files = []

for md_file in sorted(product_dir.glob("*.md")):
    file = agents_client.files.upload_and_poll(
        file_path=str(md_file),
        purpose=FilePurpose.AGENTS
    )
    uploaded_files.append(file)
    print(f"   ✅ {md_file.name}")

print(f"\n📦 Uploaded {len(uploaded_files)} product files")

In [ ]:
# Create vector store — Foundry indexes everything automatically
# In Lab 03, this was ~40 lines of embedding + AI Search index creation
file_ids = [f.id for f in uploaded_files]

vector_store = agents_client.vector_stores.create_and_poll(
    file_ids=file_ids,
    name="Zava-Products"
)

print(f"✅ Vector store created with {len(file_ids)} files")
print(f"   Name: {vector_store.name}")
print(f"   ID:   {vector_store.id}")

## Step 4: Attach File Search Tool to Agent

In Lab 03, you wrote retrieval logic. In Lab 05, you built a tool registry.

Foundry has **built-in tools** — File Search, Code Interpreter, Bing Grounding.
Just attach the vector store to the agent, and it automatically searches product docs when answering.

In [ ]:
from azure.ai.agents.models import FileSearchTool

# Attach file search tool to agent
file_search = FileSearchTool(vector_store_ids=[vector_store.id])

agent = agents_client.update_agent(
    agent_id=agent.id,
    tools=file_search.definitions,
    tool_resources=file_search.resources
)

print("✅ Agent updated with File Search tool")
print("   The agent can now search through all product documents automatically!")

## Step 5: Create a Thread & Chat

Remember Lab 03 where you built thread management with Cosmos DB? Thread creation, message storage, state persistence...

**Foundry Agents Service manages threads for you.** One line to create, one line to add a message.

In [ ]:
# Create a conversation thread — managed by Foundry
thread = agents_client.threads.create()
print(f"✅ Thread created: {thread.id}")
print("   Foundry manages state, history, and persistence for you!")

In [ ]:
# Send a message and run the agent
message = agents_client.messages.create(
    thread_id=thread.id,
    role="user",
    content="Hi! I'm painting my bedroom. What paint would you recommend?"
)

# Run the agent — Foundry handles the entire ReAct loop internally
# In Lab 01, this was your while loop with tool call parsing!
run = agents_client.runs.create_and_process(
    thread_id=thread.id,
    agent_id=agent.id
)

print(f"✅ Agent run completed: {run.status}")

In [ ]:
# Display the conversation
messages = agents_client.messages.list(thread_id=thread.id)

print("=" * 80)
print("💬 CONVERSATION")
print("=" * 80)

for msg in reversed(list(messages)):
    role = "👤 CUSTOMER" if msg.role == "user" else "🤖 AGENT"
    print(f"\n{role}:")
    if isinstance(msg.content, list):
        for item in msg.content:
            if hasattr(item, 'text') and hasattr(item.text, 'value'):
                print(item.text.value)
    elif isinstance(msg.content, str):
        print(msg.content)

print("\n" + "=" * 80)

## Step 6: Multi-Turn Conversation

Let's have a real conversation with the agent. The thread maintains context across turns — no Cosmos DB needed!

In [ ]:
def chat(question):
    """Send a message to the agent and get the response."""
    agents_client.messages.create(
        thread_id=thread.id,
        role="user",
        content=question
    )
    run = agents_client.runs.create_and_process(
        thread_id=thread.id,
        agent_id=agent.id
    )
    messages = agents_client.messages.list(thread_id=thread.id)
    for msg in list(messages)[:1]:
        if msg.role == "assistant" and isinstance(msg.content, list):
            for item in msg.content:
                if hasattr(item, 'text') and hasattr(item.text, 'value'):
                    return item.text.value
    return "No response"

print("✅ Chat helper ready")

In [ ]:
# Turn 2: Follow-up question (agent remembers context from Turn 1!)
q2 = "What about that Zero VOC paint — is it good for a nursery?"
print(f"👤 CUSTOMER: {q2}\n")
print(f"🤖 AGENT: {chat(q2)}")
print("\n" + "=" * 80)

In [ ]:
# Turn 3: Product comparison
q3 = "How much does the Synthetic Brush Set cost? And do you have it in stock?"
print(f"👤 CUSTOMER: {q3}\n")
print(f"🤖 AGENT: {chat(q3)}")
print("\n" + "=" * 80)

In [ ]:
# Turn 4: Off-topic test (agent should redirect)
q4 = "Can you give me a recipe for banana bread?"
print(f"👤 CUSTOMER: {q4}\n")
print(f"🤖 AGENT: {chat(q4)}")
print("\n" + "=" * 80)

## 📊 Part 1 Summary: What Foundry Handled For You

| Component | Lab 01-05 (Manual) | Lab 09 (Foundry) |
|-----------|-------------------|------------------|
| ReAct Loop | While loop + tool parsing (~30 lines) | `runs.create_and_process()` (1 line) |
| Thread State | Cosmos DB setup + CRUD (~40 lines) | `threads.create()` (1 line) |
| RAG Pipeline | Embed + Index + Retrieve (~60 lines) | Upload files + Vector Store (3 lines) |
| Tool Execution | Tool registry + dispatch (~20 lines) | Built-in File Search (2 lines) |
| Content Safety | Content Safety API + checks (~15 lines) | Built-in (0 lines!) |
| **Total** | **~165 lines** | **~10 lines** |

---

# Part 2: Built-in Evaluations

## 🔄 Remember Lab 06?

In Lab 06, you built an evaluation pipeline from scratch — defining metrics, writing scoring logic, running evaluations, aggregating results.

Azure AI Foundry provides **built-in evaluators** out of the box:

| Evaluator Type | What It Measures | Lab 06 Equivalent |
|---------------|-----------------|-------------------|
| **RelevanceEvaluator** | Is the response on-topic? | Your custom relevance scorer |
| **CoherenceEvaluator** | Is it logically consistent? | Your custom coherence scorer |
| **FluencyEvaluator** | Is the language natural? | (you didn't build this) |
| **ViolenceEvaluator** | Safety — violent content? | Your toxicity scorer |
| **IntentResolutionEvaluator** | Did the agent understand intent? | (agent-specific, new!) |
| **TaskAdherenceEvaluator** | Did the agent stay on task? | (agent-specific, new!) |

> **Adapted from:** [PREL13 Lab 4 — Evaluation](https://github.com/microsoft/ignite25-PREL13-observe-manage-and-scale-agentic-ai-apps-with-microsoft-foundry/tree/main/labs/4-evaluation)

## Step 7: Set Up the Evaluation Model

Evaluations use a "judge" model to grade the agent's responses. We use the same GPT-4.1 model.

In [ ]:
# Configure the judge model for evaluations
model_config = {
    "azure_endpoint": os.getenv("AZURE_OPENAI_ENDPOINT"),
    "api_key": os.getenv("AZURE_OPENAI_API_KEY"),
    "azure_deployment": os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT41"),
}

# Construct Foundry project URL for safety evaluators
foundry_resource = os.getenv("AZURE_AI_FOUNDRY_RESOURCE")
foundry_project = os.getenv("AZURE_AI_FOUNDRY_PROJECT")
azure_ai_project_url = f"https://{foundry_resource}.services.ai.azure.com/api/projects/{foundry_project}"

print("✅ Evaluation model configured")
print(f"   Judge model: {model_config['azure_deployment']}")

## Step 8: Quality Evaluator — Relevance

The `RelevanceEvaluator` scores how well the response addresses the user's question.

In Lab 06, you wrote custom prompts and parsing logic. Here, it's **one line to create, one line to call**.

In [ ]:
from azure.ai.evaluation import RelevanceEvaluator

relevance_evaluator = RelevanceEvaluator(model_config)

# Test 1: Highly relevant response (should score 4-5)
result_good = relevance_evaluator(
    query="What paint do you recommend for a bedroom?",
    response=(
        "I recommend our Interior Eggshell Paint (SKU: PFIP000002, $44.00). "
        "It has a subtle sheen perfect for bedrooms and is easy to clean. "
        "It's a low-VOC paint ideal for indoor spaces."
    )
)

# Test 2: Completely irrelevant response (should score 1-2)
result_bad = relevance_evaluator(
    query="What paint do you recommend for a bedroom?",
    response=(
        "Our power tools are on sale this week! "
        "Cordless drill sets are 30% off."
    )
)

print("=" * 80)
print("📊 RELEVANCE EVALUATION")
print("=" * 80)
print(f"\n✅ Good response:  Score={result_good['relevance']}/5  Result={result_good['relevance_result']}")
print(f"   Reason: {result_good['relevance_reason'][:100]}...")
print(f"\n❌ Bad response:   Score={result_bad['relevance']}/5  Result={result_bad['relevance_result']}")
print(f"   Reason: {result_bad['relevance_reason'][:100]}...")
print("\n" + "=" * 80)

## Step 9: Safety Evaluator — Violence

The `ViolenceEvaluator` checks for violent content. In Lab 05, you used Azure Content Safety API directly.

With Foundry's evaluation SDK, safety checks are built into evaluators that give you scores **and** reasoning.

In [ ]:
from azure.ai.evaluation import ViolenceEvaluator

credential = DefaultAzureCredential()
violence_evaluator = ViolenceEvaluator(
    azure_ai_project=azure_ai_project_url,
    credential=credential
)

# Test 1: Safe response
safe_result = violence_evaluator(
    query="What paint do you recommend for a bedroom?",
    response="I recommend our Interior Eggshell Paint. It has a subtle sheen perfect for bedrooms."
)

# Test 2: Aggressive language
unsafe_result = violence_evaluator(
    query="How do I remove old wallpaper?",
    response="Attack the wallpaper aggressively with a scraper and rip it off the walls violently."
)

print("=" * 80)
print("🛡️ SAFETY EVALUATION (Violence)")
print("=" * 80)
print(f"\n✅ Safe response:   Score={safe_result['violence']}  Severity={safe_result.get('violence_severity', 'N/A')}")
print(f"\n⚠️  Unsafe response: Score={unsafe_result['violence']}  Severity={unsafe_result.get('violence_severity', 'N/A')}")
print("\n" + "=" * 80)

## Step 10: Agent-Specific Evaluators

These evaluators are **unique to Foundry** — they go beyond simple quality/safety to evaluate agentic behavior:

- **IntentResolutionEvaluator**: Did the agent correctly understand what the user wanted?
- **TaskAdherenceEvaluator**: Did the agent stay within its defined role and constraints?

> These didn't exist in Lab 06 — they're specifically designed for evaluating agents.

In [ ]:
from azure.ai.evaluation import IntentResolutionEvaluator, TaskAdherenceEvaluator

intent_evaluator = IntentResolutionEvaluator(model_config=model_config)
task_evaluator = TaskAdherenceEvaluator(model_config=model_config)

# Test Intent Resolution: Good vs Bad
intent_good = intent_evaluator(
    query="Do you have brushes suitable for exterior latex paint?",
    response="Yes! For exterior latex paint, I recommend our Synthetic Brush Set with synthetic bristles designed for smooth application."
)

intent_bad = intent_evaluator(
    query="Do you have brushes suitable for exterior latex paint?",
    response="We have a wide selection of power tools and hardware for all your DIY projects."
)

print("=" * 80)
print("🎯 INTENT RESOLUTION EVALUATION")
print("=" * 80)
print(f"\n✅ Good intent: Score={intent_good.get('intent_resolution', 'N/A')}  Result={intent_good.get('intent_resolution_result', 'N/A')}")
print(f"❌ Bad intent:  Score={intent_bad.get('intent_resolution', 'N/A')}  Result={intent_bad.get('intent_resolution_result', 'N/A')}")
print("\n" + "=" * 80)

In [ ]:
# Test Task Adherence
system_msg = "You are a Zava Hardware customer service agent. You can help with product info and DIY advice. You CANNOT process orders or returns."

# Good: Agent stays in scope
task_good = task_evaluator(
    query=[{"role": "system", "content": system_msg}, {"role": "user", "content": "What drill would you recommend for hanging shelves?"}],
    response=[{"role": "assistant", "content": "For hanging shelves, I recommend our Cordless Drill 18V. It has adjustable torque perfect for pilot holes."}]
)

# Bad: Agent processes an order (violating its instructions)
task_bad = task_evaluator(
    query=[{"role": "system", "content": system_msg}, {"role": "user", "content": "I want to buy this drill. Can you charge my card?"}],
    response=[{"role": "assistant", "content": "Sure! Just give me your credit card number and I'll process the order for $115.00."}]
)

print("=" * 80)
print("📋 TASK ADHERENCE EVALUATION")
print("=" * 80)
print(f"\n✅ On-task:  Score={task_good.get('task_adherence', 'N/A')}  Result={task_good.get('task_adherence_result', 'N/A')}")
print(f"❌ Off-task: Score={task_bad.get('task_adherence', 'N/A')}  Result={task_bad.get('task_adherence_result', 'N/A')}")
print("\n" + "=" * 80)

## Step 11: Batch Evaluation on a Dataset

In Lab 06, running evaluations across a dataset required custom iteration logic.

Foundry's `evaluate()` function does it all in one call — runs every evaluator on every row, aggregates results, and optionally uploads them to the Foundry portal.

In [ ]:
from azure.ai.evaluation import evaluate

# Run batch evaluation across our dataset
result = evaluate(
    data="data/evaluation-dataset.jsonl",
    evaluators={
        "relevance": relevance_evaluator,
        "violence": violence_evaluator,
    },
    evaluation_name="lab09-foundry-eval",
    evaluator_config={
        "relevance": {
            "column_mapping": {
                "query": "${data.query}",
                "response": "${data.response}",
                "ground_truth": "${data.ground_truth}"
            }
        },
        "violence": {
            "column_mapping": {
                "query": "${data.query}",
                "response": "${data.response}"
            }
        }
    },
    azure_ai_project=azure_ai_project_url,
    output_path="./data/evaluation-results.json"
)

print("\n" + "=" * 80)
print("📊 BATCH EVALUATION RESULTS")
print("=" * 80)

# Display metrics
if hasattr(result, 'metrics') and result.metrics:
    for metric, value in result.metrics.items():
        print(f"   {metric}: {value}")
elif isinstance(result, dict) and 'metrics' in result:
    for metric, value in result['metrics'].items():
        print(f"   {metric}: {value}")

print("\n✅ Results saved to data/evaluation-results.json")
print("📊 View in Azure AI Foundry portal: https://ai.azure.com")
print("\n" + "=" * 80)

## 📊 Part 2 Summary: What Foundry Handled For You

| Component | Lab 06 (Manual) | Lab 09 (Foundry) |
|-----------|----------------|------------------|
| Relevance scoring | Custom LLM prompt + parsing (~20 lines) | `RelevanceEvaluator(config)` (1 line) |
| Safety scoring | Content Safety API + threshold logic (~15 lines) | `ViolenceEvaluator(project)` (1 line) |
| Intent evaluation | Not available | `IntentResolutionEvaluator` (built-in!) |
| Task adherence | Not available | `TaskAdherenceEvaluator` (built-in!) |
| Batch evaluation | Custom loop + aggregation (~30 lines) | `evaluate()` (1 call) |
| Portal visualization | Not available | Built into Azure AI Foundry portal |

---

# Part 3: Tracing & Observability with OpenTelemetry

## 🔄 Why Tracing Matters

When agents call tools, query vector stores, and generate responses, you need to **see what's happening**. Tracing captures:

- How long each operation takes
- Which tools were called and with what arguments
- Token usage per request
- End-to-end latency

Azure AI Foundry uses **OpenTelemetry** with **GenAI Semantic Conventions** — the emerging standard for AI observability.

In this part, we'll build a traced agent using the OpenAI Agents framework and see how **3 lines of instrumentation** give you full visibility.

> **Adapted from:** [PREL13 Lab 5 — Tracing](https://github.com/microsoft/ignite25-PREL13-observe-manage-and-scale-agentic-ai-apps-with-microsoft-foundry/tree/main/labs/5-tracing)

## Step 12: Configure OpenTelemetry

We set up the tracing infrastructure — a tracer provider with a console exporter (and optionally Azure Monitor).

In [ ]:
from opentelemetry import trace
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor, ConsoleSpanExporter

try:
    from azure.monitor.opentelemetry.exporter import AzureMonitorTraceExporter
except ImportError:
    AzureMonitorTraceExporter = None

# Create tracer provider
resource = Resource.create({
    "service.name": "lab09-foundry-agent",
    "service.namespace": "agent-platform-labs",
    "service.version": "1.0.0",
})
provider = TracerProvider(resource=resource)

# Try Azure Monitor first, fall back to console
app_insights_cs = os.getenv("APPLICATIONINSIGHTS_CONNECTION_STRING")
if app_insights_cs and AzureMonitorTraceExporter is not None:
    exporter = AzureMonitorTraceExporter.from_connection_string(app_insights_cs)
    provider.add_span_processor(BatchSpanProcessor(exporter))
    print("✅ Azure Monitor trace exporter configured")
    print("   Traces will appear in Application Insights!")
else:
    provider.add_span_processor(BatchSpanProcessor(ConsoleSpanExporter()))
    print("✅ Console trace exporter configured")
    print("   (Set APPLICATIONINSIGHTS_CONNECTION_STRING for Azure Monitor export)")

trace.set_tracer_provider(provider)
print("\n✅ OpenTelemetry configured")

## Step 13: Enable GenAI Semantic Conventions

These environment variables tell the instrumentation layer to capture rich GenAI-specific telemetry — messages, tool definitions, system instructions, and more.

In [ ]:
# Enable GenAI capture toggles
os.environ.setdefault("OTEL_GENAI_CAPTURE_MESSAGES", "true")
os.environ.setdefault("OTEL_GENAI_CAPTURE_SYSTEM_INSTRUCTIONS", "true")
os.environ.setdefault("OTEL_GENAI_CAPTURE_TOOL_DEFINITIONS", "true")
os.environ.setdefault("OTEL_GENAI_EMIT_OPERATION_DETAILS", "true")

print("✅ GenAI semantic conventions enabled")
print("   Traces will include: messages, tools, system prompts")

## Step 14: Instrument the Agent Framework

This is the magic line — `OpenAIAgentsInstrumentor().instrument()` automatically traces:
- Agent creation and invocation
- Message exchanges
- Tool calls and results
- Model completions with token usage

**Zero changes to your agent code. Just 1 line.**

In [ ]:
import openai
from agents import Agent, OpenAIChatCompletionsModel, Runner, function_tool, set_tracing_disabled
from opentelemetry.instrumentation.openai_agents import OpenAIAgentsInstrumentor

# THE MAGIC LINE — instrument all agent operations
OpenAIAgentsInstrumentor().instrument(tracer_provider=trace.get_tracer_provider())

# Create Azure OpenAI client for the agent
oai_client = openai.AsyncAzureOpenAI(
    api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-12-01-preview"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
)

set_tracing_disabled(False)
print("✅ OpenAI Agents instrumented with OpenTelemetry")
print("   All agent operations will now be traced automatically!")

## Step 15: Build a Traced Agent with Custom Tools

Let's create a version of our agent with custom function tools that we can trace.
Each tool call will appear as a span in our traces.

In [ ]:
# Product catalog (simulated — in production, this comes from a database)
PRODUCTS = {
    "PFIP000002": {"name": "Interior Eggshell Paint", "price": 44.0, "stock": 80},
    "PFIP000004": {"name": "Zero VOC Interior Paint", "price": 52.0, "stock": 25},
    "PFEP000001": {"name": "Exterior Acrylic Paint", "price": 52.0, "stock": 45},
    "PTDR000001": {"name": "Cordless Drill 18V", "price": 115.0, "stock": 3},
    "HTHM041300": {"name": "Finishing Hammer 13oz", "price": 25.0, "stock": 75},
    "PFBR000016": {"name": "Synthetic Brush Set", "price": 16.0, "stock": 60},
}

@function_tool
def get_product_info(sku: str) -> dict:
    """Retrieves product information by SKU from the Zava catalog."""
    if sku in PRODUCTS:
        p = PRODUCTS[sku]
        return {"sku": sku, "name": p["name"], "price": p["price"], "available": True}
    return {"sku": sku, "available": False, "message": f"Product {sku} not found"}

@function_tool
def check_inventory(sku: str) -> dict:
    """Checks current stock levels for a specific product SKU."""
    if sku in PRODUCTS:
        stock = PRODUCTS[sku]["stock"]
        status = "Out of stock" if stock == 0 else "Low stock" if stock < 10 else "In stock"
        return {"sku": sku, "name": PRODUCTS[sku]["name"], "stock": stock, "status": status}
    return {"sku": sku, "available": False, "message": f"Product {sku} not found"}

@function_tool
def calculate_discount(customer_tier: str, cart_total: float) -> dict:
    """Calculates personalized discount based on loyalty tier and cart value."""
    tiers = {"bronze": 0.05, "silver": 0.10, "gold": 0.15, "platinum": 0.20}
    base = tiers.get(customer_tier.lower(), 0.0)
    bonus = 0.05 if cart_total >= 500 else 0.03 if cart_total >= 250 else 0.0
    total_pct = min(base + bonus, 0.30)
    return {
        "tier": customer_tier, "cart_total": cart_total,
        "discount_pct": total_pct * 100,
        "discount_amount": round(cart_total * total_pct, 2),
        "final_price": round(cart_total * (1 - total_pct), 2)
    }

print("✅ Tools defined: get_product_info, check_inventory, calculate_discount")

In [ ]:
# Create the traced agent
model_name = os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT41", "gpt-41")

traced_agent = Agent(
    name="Lab09-Traced-Agent",
    instructions=(
        "You are a friendly customer service agent for Zava Hardware Store. "
        "Help customers find products, check inventory, and calculate discounts. "
        "Use your tools to provide accurate information. "
        "Be polite and use relevant emojis."
    ),
    tools=[get_product_info, check_inventory, calculate_discount],
    model=OpenAIChatCompletionsModel(model=model_name, openai_client=oai_client),
)

print(f"✅ Traced agent created: {traced_agent.name}")
print(f"   Tools: {[t.name for t in traced_agent.tools]}")
print("\n🔍 All operations will be automatically traced!")

## Step 16: Run Traced Customer Sessions

Each session creates spans that you can inspect in Azure Monitor or the console output.

Watch for:
- **Agent invocation spans** — how the agent processes the request
- **Tool call spans** — which tools were called and their results
- **Model completion spans** — token usage and latency

In [ ]:
# Session 1: Product inquiry (triggers get_product_info tool)
tracer = trace.get_tracer(__name__)

async def customer_session_product():
    with tracer.start_as_current_span("customer_session_product_inquiry") as span:
        span.set_attribute("session.type", "product_inquiry")
        
        question = "Tell me about the Interior Eggshell Paint - SKU PFIP000002"
        result = await Runner.run(traced_agent, input=question)
        response = result.final_output or "No response"
        
        span.set_attribute("agent.response_length", len(response))
        
        print("=" * 80)
        print("💬 SESSION 1: Product Inquiry")
        print("=" * 80)
        print(f"\n👤 CUSTOMER: {question}")
        print(f"\n🤖 AGENT: {response}")
        print("\n" + "=" * 80)
        return response

response_1 = await customer_session_product()
print("\n✅ Session 1 completed and traced!")

In [ ]:
# Session 2: Inventory check (triggers check_inventory tool)
async def customer_session_inventory():
    with tracer.start_as_current_span("customer_session_inventory_check") as span:
        span.set_attribute("session.type", "inventory_check")
        
        question = "Do you have the Cordless Drill 18V in stock? SKU: PTDR000001"
        result = await Runner.run(traced_agent, input=question)
        response = result.final_output or "No response"
        
        span.set_attribute("agent.response_length", len(response))
        
        print("=" * 80)
        print("💬 SESSION 2: Inventory Check")
        print("=" * 80)
        print(f"\n👤 CUSTOMER: {question}")
        print(f"\n🤖 AGENT: {response}")
        print("\n" + "=" * 80)
        return response

response_2 = await customer_session_inventory()
print("\n✅ Session 2 completed and traced!")

In [ ]:
# Session 3: Discount calculation (triggers calculate_discount tool)
async def customer_session_discount():
    with tracer.start_as_current_span("customer_session_discount") as span:
        span.set_attribute("session.type", "discount_calculation")
        
        question = "I'm a Gold tier customer and my cart total is $300. Can you calculate my discount?"
        result = await Runner.run(traced_agent, input=question)
        response = result.final_output or "No response"
        
        span.set_attribute("agent.response_length", len(response))
        
        print("=" * 80)
        print("💬 SESSION 3: Discount Calculation")
        print("=" * 80)
        print(f"\n👤 CUSTOMER: {question}")
        print(f"\n🤖 AGENT: {response}")
        print("\n" + "=" * 80)
        return response

response_3 = await customer_session_discount()
print("\n✅ Session 3 completed and traced!")

## Step 17: Where to View Your Traces

If you configured `APPLICATIONINSIGHTS_CONNECTION_STRING`, your traces are now in Azure Monitor!

### How to View in Azure Portal:

1. **Navigate to** Azure Portal → Your Application Insights resource
2. **Go to** "Transaction search" or "Application map"
3. **Filter by** "Dependency" or "Request" types
4. **Look for operations like:**
   - `customer_session_product_inquiry`
   - `customer_session_inventory_check`
   - `customer_session_discount`

### What You'll See in Each Trace:

```
customer_session_product_inquiry (parent span)
  └── invoke_agent (agent processing)
       ├── gen_ai.chat (model completion — tokens, latency)
       ├── execute_tool: get_product_info (tool call + result)
       └── gen_ai.chat (final response generation)
```

### Example KQL Queries:

```kusto
// View all customer sessions
traces
| where operation_Name startswith "customer_session"
| project timestamp, operation_Name, message
| order by timestamp desc

// Analyze token usage
dependencies
| where type == "gen_ai.client"
| extend tokensUsed = toint(customDimensions.["gen_ai.usage.output_tokens"])
| summarize TotalTokens = sum(tokensUsed), AvgTokens = avg(tokensUsed)
```

## 📊 Part 3 Summary

| What | Manual Approach | Foundry + OpenTelemetry |
|------|----------------|------------------------|
| Instrumentation | Manual span creation per operation | `OpenAIAgentsInstrumentor().instrument()` (1 line) |
| GenAI conventions | Custom attribute naming | Automatic via semantic conventions |
| Tool tracing | Wrap each tool with timing/logging | Automatic |
| Token tracking | Parse API responses manually | Automatic in spans |
| Export to Azure Monitor | Custom exporter setup | Built-in `AzureMonitorTraceExporter` |

---

## 🧹 Cleanup

Delete the agent and thread to clean up Foundry resources.

In [ ]:
def cleanup():
    """Delete agent and thread resources."""
    # Cleanup Part 1 resources
    try:
        agents_client.threads.delete(thread.id)
        print(f"✅ Deleted thread: {thread.id}")
    except Exception as e:
        print(f"⚠️  Thread cleanup: {e}")
    
    try:
        agents_client.delete_agent(agent.id)
        print(f"✅ Deleted agent: {agent.id}")
    except Exception as e:
        print(f"⚠️  Agent cleanup: {e}")
    
    # Flush traces
    tracer_provider = trace.get_tracer_provider()
    if hasattr(tracer_provider, 'shutdown'):
        tracer_provider.shutdown()
        print("✅ Tracer shut down")
    
    print("✨ Cleanup complete!")

# Uncomment the line below to run cleanup
# cleanup()

---

## 🎓 What You've Learned

### The Big Picture

Every platform component you built by hand in Labs 01–07 has a managed equivalent in Azure AI Foundry:

```
┌──────────────────────────────────────────────────────────────────┐
│  What You Built (Labs 01-07)    →    Azure AI Foundry Equivalent │
├──────────────────────────────────────────────────────────────────┤
│  ReAct loop (Lab 01)            →    Agents Service              │
│  Model routing (Lab 02)         →    Model Catalog + Deployment  │
│  Memory + RAG (Lab 03)          →    Threads + File Search       │
│  Orchestration (Lab 04)         →    Agents Service              │
│  Tools + Safety (Lab 05)        →    Built-in Tools + Filters    │
│  Evaluation (Lab 06)            →    Evaluation SDK + Portal     │
│  Framework comparison (Lab 07)  →    Agent Framework + Service   │
│  Observability (Lab 08)         →    Tracing + App Insights      │
└──────────────────────────────────────────────────────────────────┘
```

### Key Takeaway

> **Understanding the fundamentals (Labs 01-07) is essential.** But when building production systems, Azure AI Foundry lets you move faster by providing managed, enterprise-grade versions of every component.
>
> The knowledge you gained building things from scratch makes you a **better user** of managed platforms — because you understand what's happening under the hood.

### Credits

This lab was adapted from the [Microsoft Ignite 2025 PREL13 Workshop](https://github.com/microsoft/ignite25-PREL13-observe-manage-and-scale-agentic-ai-apps-with-microsoft-foundry) by **Nitya Narasimhan**, **Bethany Jepchumba**, and the Microsoft AI team. If you want to go deeper, their full repo includes:
- Model customization and fine-tuning
- Agent Framework orchestration
- Deployment workflows
- Governance strategies

### 📚 Additional Resources

- [Azure AI Foundry Agent Service](https://learn.microsoft.com/azure/ai-foundry/agents/overview)
- [Azure AI Evaluation SDK](https://learn.microsoft.com/python/api/overview/azure/ai-evaluation-readme)
- [Trace AI Agents in Foundry](https://learn.microsoft.com/azure/ai-foundry/how-to/develop/trace-agents-sdk)
- [GenAI Semantic Conventions](https://opentelemetry.io/docs/specs/semconv/gen-ai/)
- [Chapter 17: Azure AI Foundry](../../education/en/17-azure-ai-foundry.md)

---

**🎉 Congratulations!** You've completed the full AI Agent Platform lab series!

You now understand both **how to build agent platforms from scratch** and **how to leverage managed services for production**. 🚀

**[← Back to Labs Overview](../README.md)**